In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 - Silver Layer: Clean, Standardise, De-duplicate
# MAGIC
# MAGIC **Goal:** turn raw Bronze rows into a trustworthy, query-ready `silver_transactions`
# MAGIC table with consistent types, no duplicates, and clear handling of bad records.
# MAGIC
# MAGIC Techniques demonstrated:
# MAGIC - Incremental read of Bronze via Structured Streaming (`readStream.table(...)`)
# MAGIC - **Data quality gate**: required-field / range / lookup checks, with rejects routed to
# MAGIC   a `silver_quarantine` table rather than silently dropped
# MAGIC - **De-duplication** across files/batches (not just within one micro-batch) using a
# MAGIC   `MERGE` upsert keyed on `transaction_id` -> naturally **idempotent**, safe to re-run
# MAGIC   or reprocess historical files without creating duplicates
# MAGIC - Simple business logic / standardisation (currency casing, net amount after discount,
# MAGIC   derived date parts, refund flag)
# MAGIC - Change Data Feed enabled on Silver so Gold can consume **only what changed**

# COMMAND ----------

from pyspark.sql import functions as F, types as T

CATALOG = "lakehouse_demo"
SCHEMA = "transactions"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_quarantine"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

VALID_CURRENCIES = ["GBP", "USD", "EUR"]
VALID_STATUSES = ["completed", "refunded", "failed"]

# COMMAND ----------

# MAGIC %md
# MAGIC ## Create Silver + Quarantine tables up front (so MERGE has a target, and so the schema
# MAGIC is explicit and enforced rather than left to inference)

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    transaction_id      STRING NOT NULL,
    customer_id         STRING NOT NULL,
    transaction_ts      TIMESTAMP NOT NULL,
    transaction_date    DATE NOT NULL,
    transaction_month   STRING NOT NULL,
    amount              DOUBLE NOT NULL,
    discount_pct        DOUBLE,
    net_amount          DOUBLE NOT NULL,
    currency            STRING NOT NULL,
    category            STRING,
    payment_method      STRING,
    store_id            STRING,
    status              STRING NOT NULL,
    is_refund           BOOLEAN NOT NULL,
    _source_file         STRING,
    _bronze_ingest_ts     TIMESTAMP,
    _silver_processed_ts  TIMESTAMP
)
USING DELTA
PARTITIONED BY (transaction_month)
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
    transaction_id   STRING,
    customer_id      STRING,
    amount           DOUBLE,
    currency         STRING,
    status           STRING,
    reject_reasons   ARRAY<STRING>,
    _source_file      STRING,
    _quarantined_ts   TIMESTAMP
)
USING DELTA
""")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Transformation logic
# MAGIC Applied once per micro-batch inside `foreachBatch`, which lets us:
# MAGIC 1. split good vs bad records,
# MAGIC 2. de-duplicate *within* the batch,
# MAGIC 3. `MERGE` the good records into Silver (idempotent upsert on `transaction_id`),
# MAGIC 4. append the bad records to the quarantine table for triage.

# COMMAND ----------

def clean_and_standardise(bronze_batch_df):
    df = bronze_batch_df

    # Bronze stores amount/discount as whatever the source format produced (string from CSV,
    # double from JSON) - cast defensively here rather than assuming.
    df = (
        df.withColumn("customer_id", F.trim(F.col("customer_id")))
          .withColumn("currency", F.upper(F.trim(F.col("currency"))))
          .withColumn("status", F.lower(F.trim(F.col("status"))))
          .withColumn("amount", F.col("amount").cast(T.DoubleType()))
          .withColumn(
              "discount_pct",
              F.col("discount_pct").cast(T.DoubleType()) if "discount_pct" in df.columns
              else F.lit(None).cast(T.DoubleType())
          )
          .withColumn("transaction_ts", F.to_timestamp("transaction_ts"))
    )

    # --- Data quality rules -------------------------------------------------
    reasons = F.array_compact(F.array(
        F.when(F.col("transaction_id").isNull(), F.lit("missing_transaction_id")),
        F.when(F.col("customer_id").isNull(), F.lit("missing_customer_id")),
        F.when(F.col("transaction_ts").isNull(), F.lit("missing_or_unparseable_timestamp")),
        F.when(F.col("amount").isNull(), F.lit("missing_amount")),
        F.when((F.col("amount").isNotNull()) & (F.col("amount") <= 0) & (F.col("status") != "refunded"),
               F.lit("non_positive_amount")),
        F.when(~F.col("currency").isin(VALID_CURRENCIES), F.lit("invalid_currency")),
        F.when(~F.col("status").isin(VALID_STATUSES), F.lit("invalid_status")),
    ))

    df = df.withColumn("reject_reasons", reasons)

    bad_df = df.filter(F.size("reject_reasons") > 0).select(
        "transaction_id", "customer_id", "amount", "currency", "status",
        "reject_reasons", "_source_file"
    ).withColumn("_quarantined_ts", F.current_timestamp())

    good_df = df.filter(F.size("reject_reasons") == 0)

    # --- De-duplicate within the micro-batch --------------------------------
    # Keep the most recently ingested version of each transaction_id if the same file (or a
    # re-sent file) contains the id more than once.
    w = F.row_number().over(
        __import__("pyspark").sql.Window.partitionBy("transaction_id").orderBy(F.col("_ingest_ts").desc())
    )
    good_df = good_df.withColumn("_rn", w).filter(F.col("_rn") == 1).drop("_rn")

    # --- Business logic / standardisation ------------------------------------
    good_df = (
        good_df
        .withColumn("net_amount", F.round(F.col("amount") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0)) / 100), 2))
        .withColumn("transaction_date", F.to_date("transaction_ts"))
        .withColumn("transaction_month", F.date_format("transaction_ts", "yyyy-MM"))
        .withColumn("is_refund", F.col("status") == F.lit("refunded"))
        .withColumnRenamed("_ingest_ts", "_bronze_ingest_ts")
        .withColumn("_silver_processed_ts", F.current_timestamp())
        .select(
            "transaction_id", "customer_id", "transaction_ts", "transaction_date", "transaction_month",
            "amount", "discount_pct", "net_amount", "currency", "category", "payment_method",
            "store_id", "status", "is_refund", "_source_file", "_bronze_ingest_ts", "_silver_processed_ts"
        )
    )

    return good_df, bad_df

# COMMAND ----------

def process_batch(batch_df, batch_id):
    good_df, bad_df = clean_and_standardise(batch_df)

    good_df.createOrReplaceTempView("silver_updates")
    # MERGE keyed on transaction_id -> idempotent upsert. Re-running this stream, or
    # reprocessing a previously-seen bronze batch, will never create duplicate rows: it will
    # just overwrite the row with itself (or an updated version if it changed upstream).
    batch_df.sparkSession.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    bad_df.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)

    good_count = good_df.count()
    bad_count = bad_df.count()
    print(f"[batch {batch_id}] merged {good_count} good rows, quarantined {bad_count} bad rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Run the stream (incremental, checkpointed)

# COMMAND ----------

bronze_stream = spark.readStream.table(BRONZE_TABLE)

query = (
    bronze_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/silver_transactions")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sanity checks

# COMMAND ----------

display(spark.sql(f"SELECT count(*) AS silver_rows FROM {SILVER_TABLE}"))
display(spark.sql(f"SELECT count(*) AS quarantined_rows FROM {QUARANTINE_TABLE}"))

# COMMAND ----------

display(spark.sql(f"""
    SELECT reject_reasons, count(*) AS n
    FROM {QUARANTINE_TABLE}
    GROUP BY reject_reasons
    ORDER BY n DESC
"""))

# COMMAND ----------

# Confirm no duplicate transaction_ids made it through
display(spark.sql(f"""
    SELECT transaction_id, count(*) AS n
    FROM {SILVER_TABLE}
    GROUP BY transaction_id
    HAVING count(*) > 1
"""))

# COMMAND ----------

spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (customer_id)")

In [0]:
%sql 
SELECT count(*) FROM lakehouse_demo.transactions.bronze_transactions;

In [0]:
%sql 
SELECT count(*) FROM lakehouse_demo.transactions.silver_quarantine;

In [0]:
%sql
SELECT * FROM lakehouse_demo.transactions.silver_transactions;

In [0]:
dbutils.fs.ls("/Volumes/lakehouse_demo/transactions/checkpoints/silver_transactions")

In [0]:
dbutils.fs.rm("/Volumes/lakehouse_demo/transactions/checkpoints/silver_transactions", recurse=True)

In [0]:
df = spark.read.table("lakehouse_demo.transactions.bronze_transactions")
print(df.count())
df.printSchema()
display(df.limit(5))

In [0]:
dbutils.fs.rm("/Volumes/lakehouse_demo/transactions/checkpoints/silver_transactions", recurse=True)

In [0]:
def process_batch(batch_df, batch_id):
    good_df, bad_df = clean_and_standardise(batch_df)

    good_df.createOrReplaceTempView("silver_updates")
    # MERGE keyed on transaction_id -> idempotent upsert. Re-running this stream, or
    # reprocessing a previously-seen bronze batch, will never create duplicate rows: it will
    # just overwrite the row with itself (or an updated version if it changed upstream).
    batch_df.sparkSession.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING silver_updates AS source
        ON target.transaction_id = source.transaction_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    bad_df.write.format("delta").mode("append").saveAsTable(QUARANTINE_TABLE)

    good_count = good_df.count()
    bad_count = bad_df.count()
    print(f"[batch {batch_id}] merged {good_count} good rows, quarantined {bad_count} bad rows")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Run the stream (incremental, checkpointed)

# COMMAND ----------

bronze_stream = spark.readStream.table(BRONZE_TABLE)

query = (
    bronze_stream.writeStream
    .foreachBatch(process_batch)
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/silver_transactions")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()


In [0]:
print(query.lastProgress)
print(query.recentProgress)

In [0]:
bronze_static = spark.table("lakehouse_demo.transactions.bronze_transactions")
process_batch(bronze_static, 0)

In [0]:
import inspect
print(inspect.getsource(clean_and_standardise))

In [0]:
bronze_df = spark.table("lakehouse_demo.transactions.bronze_transactions")
good_df, bad_df = clean_and_standardise(bronze_df)
print("good:", good_df.count(), "bad:", bad_df.count())

In [0]:
dbutils.fs.rm("/Volumes/lakehouse_demo/transactions/checkpoints/silver_transactions", recurse=True)